In [1]:
import polars as pl 
import pandas as pd
import numpy as np
import math
import polars.selectors as cs
import altair as alt

# Análisis de Complejidad para Zonas Metropolitanas de México

<a href="https://colab.research.google.com/github/milocortes/ciencia_datos_avanzada/blob/edicion-2024/notebooks/variables_data_types_mide_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Se proporcionan los datos de conteos de Unidades Económicas por rama de actividad del SCIAN y por Zona Metropolitana del Censo Económico de 2023 del INEGI.

Los datos se encuentran en la ruta del repositorio `datos/complexity_zm_2023.csv`. El esquema de los datos es el siguiente:
* `zm` : Columna de Zona Metropolitana.
* `rama_id` : Clave de la actividad a nivel de rama.
* `industria` : Nombre de la clave de actividad.
* `transable` : La columna toma valor igual a 1 si la actividad es transable.
* `ue` : Cantidad de Unidades Económicas.

Con estos datos se solita realizar un análisis de complejidad para responder las siguientes preguntas:

/// admonition | Preguntas a responder.

* ¿Cuáles son las 10 Zonas Metropolitanas más Complejas?
* ¿Cuáles son las 10 Industrias más Complejas?
///

Para responder a dichas preguntas, necesitas calcular los siguientes requerimientos:

/// admonition | Secuencia de cálculos

* Calcula RCA.
* Calcula Matriz de Especialización Binaria $M$.
* Calcula Matriz de Proximidad de las actividades industriales.
* Calcula Diversidad y su respectiva matriz diagonal $D$.
* Calcula Ubicuidad y su respectiva matriz diagonal $U$.
* Calcula Densidad.
* Calcula Distancia.
* Calcula matrices de diversidad-ponderada $\tilde{M}$ para zonas metropolitanas e industrias.
* Calcula los índices de Complejidad Económica (ICE) y Complejidad de las Industrias (ICI). Para esto, necesitas calcular lo siguiente:
    *  Calcular los eigenvectores y eigenvalores de las matrices $\tilde{M}$ de zonas metropolitanas e industrias.
    *  Ajusta el signo del ECI e ICI.
    *  Normaliza el ECI e ICI.
///

In [2]:
## Cargamos datos
datos = pd.read_csv("../datos/complexity_zm_2023.csv")
## Convertimos datos a polars
datos = pl.from_pandas(datos)

In [3]:
## Calculamos RCA
datos_rca = datos.with_columns(
    rca = (
        pl.col("ue")/pl.col("ue").sum().over("zm")
    ) /
    (
        pl.col("ue").sum().over("rama_id")/pl.col("ue").sum()
    )

).with_columns(
    pl.col("rama_id").cast(pl.Int64)
)

datos_rca

zm,rama_id,industria,transable,ue,rca
str,i64,str,i64,i64,f64
"""Zona metropolitana de Aguascal…",1125,"""Acuicultura""",1,5,0.367049
"""Zona metropolitana de Aguascal…",1141,"""Pesca""",1,9,0.131453
"""Zona metropolitana de Aguascal…",1151,"""Servicios relacionados con la …",0,10,1.413024
"""Zona metropolitana de Aguascal…",1152,"""Servicios relacionados con la …",0,2,2.294368
"""Zona metropolitana de Aguascal…",1153,"""Servicios relacionados con el …",0,0,0.0
…,…,…,…,…,…
"""Manzanillo, Col.""",8123,"""Servicios funerarios y adminis…",0,16,1.433508
"""Manzanillo, Col.""",8124,"""Estacionamientos y pensiones p…",0,15,0.445605
"""Manzanillo, Col.""",8129,"""Servicios de revelado e impres…",0,4,0.257514


In [4]:
## Calculamos matriz de especialización binaria
## Umbral de RCA
rca_umbral = 1.0

datos_rca_m = datos_rca.with_columns(
    M = pl.when(
        pl.col("rca")>= rca_umbral   
    ).then(
        pl.lit(1)
    ).otherwise(
        pl.lit(0)
    )
)
datos_rca_m

zm,rama_id,industria,transable,ue,rca,M
str,i64,str,i64,i64,f64,i32
"""Zona metropolitana de Aguascal…",1125,"""Acuicultura""",1,5,0.367049,0
"""Zona metropolitana de Aguascal…",1141,"""Pesca""",1,9,0.131453,0
"""Zona metropolitana de Aguascal…",1151,"""Servicios relacionados con la …",0,10,1.413024,1
"""Zona metropolitana de Aguascal…",1152,"""Servicios relacionados con la …",0,2,2.294368,1
"""Zona metropolitana de Aguascal…",1153,"""Servicios relacionados con el …",0,0,0.0,0
…,…,…,…,…,…,…
"""Manzanillo, Col.""",8123,"""Servicios funerarios y adminis…",0,16,1.433508,1
"""Manzanillo, Col.""",8124,"""Estacionamientos y pensiones p…",0,15,0.445605,0
"""Manzanillo, Col.""",8129,"""Servicios de revelado e impres…",0,4,0.257514,0


In [5]:
## Obtenemos la matriz M en formato dataframe
M_df = datos_rca_m.pivot("rama_id", 
                  index = "zm", 
                  values = "M"
            ).fill_null(0).sort("zm")
M_df

zm,1125,1141,1151,1152,1153,2111,2121,2122,2123,2131,2211,2212,2213,2361,2362,2371,2372,2373,2379,2381,2382,2383,2389,3111,3112,3113,3114,3115,3116,3117,3118,3119,3121,3122,3131,3132,…,6221,6222,6223,6231,6232,6233,6239,6241,6242,6243,6244,7111,7112,7113,7114,7115,7121,7131,7132,7139,7211,7212,7213,7223,7224,7225,8111,8112,8113,8114,8121,8122,8123,8124,8129,8131,8132
str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,…,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32
"""Manzanillo, Col.""",1,1,0,0,0,0,0,1,1,0,0,0,0,1,1,1,1,1,1,1,1,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,…,0,1,1,1,0,0,1,1,0,1,1,0,1,1,0,0,0,0,1,0,1,0,1,0,0,1,0,0,1,1,0,1,1,0,0,1,1
"""Metrópoli municipal de Acapulc…",1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,1,1,0,1,0,0,0,…,0,0,0,1,0,0,1,1,0,1,0,0,1,0,0,1,0,1,0,0,1,1,1,0,1,1,0,1,0,0,0,0,1,0,0,1,1
"""Metrópoli municipal de Campech…",0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,0,1,0,1,0,1,0,0,0,1,0,0,…,0,0,0,1,0,1,1,0,0,0,1,0,1,0,0,1,0,0,1,1,1,0,1,1,0,1,0,1,0,1,0,0,0,0,1,1,1
"""Metrópoli municipal de Chetuma…",0,1,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,1,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,1,1,0,1,1,1,1,1,0,1,1,1,1,0,1,0,1,0,0,0,1,1
"""Metrópoli municipal de Ciudad …",0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,1,1,1,0,1,1,0,1,0,1,0,1,0,0,0,0,0,1,1,0,0,1,1,1,1,1,1,0,0,1,0,1,0,0,0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zona metropolitana de Villaher…",1,1,1,1,0,1,0,0,0,1,0,0,0,1,0,1,1,1,1,1,0,0,1,0,0,0,0,0,1,0,1,0,0,0,1,0,…,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,0,0,1,0,1,0,1,1,0,1,1,1,1,1,0,1,1,0,0,1,0
"""Zona metropolitana de Xalapa""",1,0,0,0,1,0,0,0,0,0,0,0,1,0,1,1,1,1,1,1,0,0,0,0,0,1,1,0,0,0,1,1,1,1,0,0,…,0,0,0,1,1,0,0,1,1,1,0,0,0,1,0,1,1,1,0,1,1,1,1,1,0,1,0,1,0,1,1,1,1,1,1,1,1
"""Zona metropolitana de Zacateca…",0,0,0,0,0,0,0,1,1,1,1,0,1,1,1,1,1,1,0,0,0,1,1,0,0,1,0,1,0,0,0,1,1,0,0,0,…,0,1,0,1,1,0,0,1,0,0,1,1,0,1,0,1,1,0,0,1,1,0,0,1,0,1,1,1,0,0,0,0,1,0,0,1,1


In [6]:
## Obtenemos la matriz M como arreglo de numpy
M = M_df.select(
    pl.exclude("zm")
).to_numpy()

M

array([[1, 1, 0, ..., 0, 1, 1],
       [1, 1, 0, ..., 0, 1, 1],
       [0, 1, 0, ..., 1, 1, 1],
       ...,
       [0, 0, 0, ..., 0, 1, 1],
       [0, 0, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 1, 0, 0]], shape=(95, 278), dtype=int32)

In [7]:
## Calculamos diversidad
diversidad = M.sum(axis = 1)
D = np.diag(diversidad)

## Calculamos ubicuidad
ubicuidad = M.sum(axis = 0)
U = np.diag(ubicuidad)

In [8]:
## Calculamos matriz de Proximidad
proximity = M.T @ M / ubicuidad[np.newaxis, :]  
proximity = np.minimum(proximity, proximity.T)
proximity = np.nan_to_num(proximity)
proximity

array([[1.        , 0.62962963, 0.325     , ..., 0.23076923, 0.31343284,
        0.34042553],
       [0.62962963, 1.        , 0.375     , ..., 0.22222222, 0.32835821,
        0.36170213],
       [0.325     , 0.375     , 1.        , ..., 0.325     , 0.46268657,
        0.5106383 ],
       ...,
       [0.23076923, 0.22222222, 0.325     , ..., 1.        , 0.34328358,
        0.29787234],
       [0.31343284, 0.32835821, 0.46268657, ..., 0.34328358, 1.        ,
        0.64179104],
       [0.34042553, 0.36170213, 0.5106383 , ..., 0.29787234, 0.64179104,
        1.        ]], shape=(278, 278))

In [9]:
## Calculamos Densidad
density = (np.dot(M,proximity)/np.sum(proximity, axis=1))
density = np.nan_to_num(density)
density

array([[0.4245088 , 0.45510914, 0.39549377, ..., 0.34062753, 0.43582668,
        0.433018  ],
       [0.29924894, 0.31430656, 0.24385123, ..., 0.20130535, 0.270112  ,
        0.2672857 ],
       [0.39383383, 0.43961083, 0.40115784, ..., 0.36463884, 0.44463752,
        0.44125517],
       ...,
       [0.45644006, 0.45687724, 0.48911396, ..., 0.42609513, 0.54104595,
        0.52594695],
       [0.32406572, 0.3160611 , 0.35742457, ..., 0.27961056, 0.3564673 ,
        0.33964732],
       [0.23509118, 0.21360654, 0.22381815, ..., 0.31441434, 0.20935294,
        0.21695749]], shape=(95, 278))

In [10]:
## Calculamos Distancia
distancia = (np.dot((1 - M),proximity)/np.sum(proximity, axis=1))
distancia = np.nan_to_num(distancia)
distancia

array([[0.5754912 , 0.54489086, 0.60450623, ..., 0.65937247, 0.56417332,
        0.566982  ],
       [0.70075106, 0.68569344, 0.75614877, ..., 0.79869465, 0.729888  ,
        0.7327143 ],
       [0.60616617, 0.56038917, 0.59884216, ..., 0.63536116, 0.55536248,
        0.55874483],
       ...,
       [0.54355994, 0.54312276, 0.51088604, ..., 0.57390487, 0.45895405,
        0.47405305],
       [0.67593428, 0.6839389 , 0.64257543, ..., 0.72038944, 0.6435327 ,
        0.66035268],
       [0.76490882, 0.78639346, 0.77618185, ..., 0.68558566, 0.79064706,
        0.78304251]], shape=(95, 278))

In [11]:
# Calculamos M tilde cc
M_tilde_cc = np.linalg.pinv(D) @ M @ np.linalg.pinv(U) @ M.T
M_tilde_cc 

array([[0.0284085 , 0.00985198, 0.0130921 , ..., 0.01227745, 0.00767507,
        0.00570445],
       [0.01469448, 0.03016065, 0.0131113 , ..., 0.0104158 , 0.0099935 ,
        0.00591194],
       [0.0130921 , 0.00879053, 0.02757405, ..., 0.01225423, 0.00791305,
        0.00485814],
       ...,
       [0.00964657, 0.00548689, 0.00962832, ..., 0.0286304 , 0.00950601,
        0.00748874],
       [0.00854944, 0.0074635 , 0.00881454, ..., 0.01347687, 0.03048347,
        0.00860272],
       [0.00583711, 0.00405587, 0.00497112, ..., 0.00975278, 0.0079025 ,
        0.07957896]], shape=(95, 95))

In [12]:
# Calculamos M tilde pp
M_tilde_pp = np.linalg.pinv(U) @ M.T @ np.linalg.pinv(D) @ M
M_tilde_pp 

array([[0.01186955, 0.00751196, 0.00524338, ..., 0.00279678, 0.00944506,
        0.00692452],
       [0.00695552, 0.012803  , 0.0060444 , ..., 0.00225753, 0.00922652,
        0.00728122],
       [0.00327711, 0.00407997, 0.01142383, ..., 0.00316967, 0.00802743,
        0.00608751],
       ...,
       [0.00268921, 0.00234435, 0.00487641, ..., 0.01044432, 0.00924238,
        0.00505284],
       [0.00352428, 0.00371815, 0.0047925 , ..., 0.00358659, 0.01137789,
        0.00710276],
       [0.00368326, 0.00418283, 0.00518086, ..., 0.00279519, 0.01012521,
        0.01101458]], shape=(278, 278))

In [13]:
## Calculamos los eigenvectores y eigenvalores de la matriz M tilde cc
eigenvalues_cc, eigenvectors_cc = np.linalg.eig(M_tilde_cc)

## Obtenemos el eigenvector asociado con el segundo eigenvalor más grande
Kc = eigenvectors_cc[:, 1]

Kc

array([-0.02706793,  0.09478377, -0.00593797,  0.00790598, -0.14194502,
       -0.04581234, -0.00270882, -0.06648997, -0.0182207 , -0.06239473,
        0.02858207, -0.0717657 , -0.02258293, -0.02799845, -0.02848648,
       -0.02067183, -0.10001111, -0.02804661, -0.11928664, -0.10431456,
       -0.07559802, -0.01218532,  0.12189178,  0.09473583,  0.08099062,
        0.1172499 ,  0.11553314, -0.01632283, -0.00282991,  0.21129467,
        0.18358126,  0.26845208,  0.04369638,  0.07412528,  0.20436536,
        0.07071021,  0.14511977,  0.10694652, -0.01420775,  0.11915467,
        0.10873273, -0.00848565,  0.11284589,  0.21221105,  0.16161453,
        0.23010167,  0.22569674, -0.04486637, -0.07263539, -0.01033676,
       -0.10149368,  0.1652292 , -0.00865272, -0.01832764,  0.13108302,
        0.05532956,  0.04555282, -0.03214767, -0.09471103, -0.04602339,
       -0.06157219,  0.09812336, -0.01269542,  0.10918645, -0.06576454,
       -0.12942678,  0.02968245, -0.04239239,  0.09753355,  0.12

In [14]:
## Calculamos los eigenvectores y eigenvalores de la matriz M tilde cc
eigenvalues_pp, eigenvectors_pp = np.linalg.eig(M_tilde_pp)

## Obtenemos el eigenvector asociado con el segundo eigenvalor más grande
Kp = eigenvectors_pp[:, 1].astype(float)

Kp 

/tmp/ipykernel_1681433/1790232361.py:5: ComplexWarning: Casting complex values to real discards the imaginary part
  Kp = eigenvectors_pp[:, 1].astype(float)


array([-4.00774175e-02, -3.31523917e-02, -3.65639792e-03,  1.29880799e-02,
        2.55917737e-02,  5.62766740e-02,  1.01804007e-01,  9.31942756e-03,
       -3.10716034e-02,  5.32588382e-02, -1.74652223e-02,  1.24281074e-01,
       -1.05241882e-01,  4.57280955e-02,  4.37283939e-02,  4.11193988e-03,
        9.24139339e-03, -3.86905945e-03,  5.62260389e-03,  2.36661606e-02,
        7.26774746e-02,  3.71937952e-02,  4.51088942e-02, -1.28130588e-02,
       -8.62973038e-02, -7.59471057e-02, -6.17789470e-02, -6.38310576e-02,
       -7.38741304e-02,  2.70838211e-02, -1.07191667e-01, -8.60556592e-02,
        7.56095024e-03,  1.40323780e-02, -1.51780529e-01, -1.65969752e-01,
       -8.05549487e-02, -1.15214794e-01, -2.04802803e-01, -1.99373896e-01,
       -1.34549683e-01, -7.40390640e-02, -7.87855540e-02, -1.93595704e-01,
       -1.29415924e-01, -9.92075109e-02, -4.97075253e-03, -5.94333762e-02,
        3.10845806e-02, -1.09207959e-04,  2.39076852e-02,  1.96222270e-02,
       -5.26525299e-03,  

In [15]:
## Adjust sign of ECI and PCI so it makes sense, as per book
corr_mat = np.corrcoef(diversidad, Kc)
s1 = math.copysign(1.0, corr_mat[0,1])

In [16]:
## Ajustamos y normalizamos ECI y PCI
# La normalización usando usando la media y std ECI preserva
# Que ---> ECI = (mean of PCI of products for which MCP=1)

eci = s1*(Kc - np.mean(Kc))/np.std(Kc)
pci = (Kp - np.mean(Kp))/np.std(Kp)

In [17]:
### Reunimos los datos calculados 
df_eci = pl.DataFrame(
      {
        "zm" : M_df["zm"], 
        "eci" : eci, 
    }  
)

df_pci = pl.DataFrame(
      {
        "rama_id" : [int(i) for i in M_df.columns[1:]],
        "pci" : pci
    }  
).join(
    datos.select("rama_id", "industria").unique(), 
    on = "rama_id", 
    how = "inner"
)

In [18]:
df_eci

zm,eci
str,f64
"""Manzanillo, Col.""",0.546806
"""Metrópoli municipal de Acapulc…",-0.684415
"""Metrópoli municipal de Campech…",0.333304
"""Metrópoli municipal de Chetuma…",0.193421
"""Metrópoli municipal de Ciudad …",1.707554
…,…
"""Zona metropolitana de Villaher…",0.33204
"""Zona metropolitana de Xalapa""",-0.124257
"""Zona metropolitana de Zacateca…",0.244649


In [19]:
df_pci

rama_id,pci,industria
i64,f64,str
4684,-0.056001,"""Comercio al por menor de combu…"
3241,0.213331,"""Fabricación de productos deriv…"
3261,1.35064,"""Fabricación de productos de pl…"
6111,0.703,"""Escuelas de educación básica, …"
2211,-0.40919,"""Generación, transmisión, distr…"
…,…,…
7213,-0.790566,"""Pensiones y casas de huéspedes…"
3221,0.405728,"""Fabricación de pulpa, papel y …"
4859,-1.276545,"""Otro transporte terrestre de p…"


# Oportunidades Potenciales de Crecimiento

Definen tres estrategias de diversificación en los que se ponderan de forma distinta las tres medidas:

* Low-hanging Fruit:
    * Densidad : 60%
    * Complejidad : 15%
    * Ganancia de Oportunidad : 25
* Balanced Portfolio:
    * Densidad : 50%
    * Complejidad : 15%
    * Ganancia de Oportunidad : 35
* Long Jumps:
    * Densidad : 45%
    * Complejidad : 20%
    * Ganancia de Oportunidad : 35

Para cada portafolio, indica las 10 industrias que recomendarías promover a la Zona Metropolitana de tu elección.

/// attention | Atención!

* Calcula ganancia de oportunidad.
///

# Opportunity Gain - Opportunity Outlook Gain (COG)

- Podemos utilizar el opportunity value para calcular el beneficio potencial que obtendría un lugar si se especializa en una nueva actividad particular.
- Se llama a este valor como la **ganancia de oportunidad (Opportunity Gain)** que un lugar $c$ obtendría de especializarse en la actividad $p$.
- Se calcula como el cambio en el valor de oportunidad obtenido de especializarse en la actividad $p$.
- La ganancia de oportunidad cuantifica la contribución de especializarse en una nueva actividad en términos de abrir las puertas a actividades cada vez más complejas.
- Formalmente la calculamos como:

\begin{equation}
\text { opportunity gain }_c=\sum_{p^{\prime}} \frac{\phi_{p p^{\prime}}}{\sum_{p^{\prime \prime}} \phi_{p^{\prime \prime} p^{\prime}}}\left(1-M_{c p^{\prime}}\right) P C I_{p^{\prime}}
\end{equation}

>En la publicación **The atlas of economic complexity: Mapping paths to prosperity** se especifica la siguiente fórmula para el Opportunity Gain
>
\begin{equation}
\text { opportunity gain }_c=\sum_{p^{\prime}} \frac{\phi_{p p^{\prime}}}{\sum_{p^{\prime \prime}} \phi_{p^{\prime \prime} p^{\prime}}}\left(1-M_{c p^{\prime}}\right) P C I_{p^{\prime}}-\left(1-d_{c p}\right) P C I_p
\end{equation}

> Aquí usaremos la especificación del glosario del portal del Atlas de Complejidad Económica ([liga](https://atlas.hks.harvard.edu/glossary))

In [20]:
## Calculamos ganancia de oportunidad
og = (proximity  @ ((proximity.sum(axis=1)**-1)[np.newaxis].T * (1 - M.T) * (pci[np.newaxis].T))).T
og

array([[-0.13696219, -0.11092226, -0.06801999, ...,  0.1107084 ,
        -0.09044475, -0.03616829],
       [-0.01440355,  0.00437902,  0.06662698, ...,  0.21268442,
         0.02959706,  0.09115824],
       [-0.11942058, -0.08316184, -0.03048211, ...,  0.13268187,
        -0.06272158, -0.01051117],
       ...,
       [-0.09785587, -0.07978381, -0.00035954, ...,  0.1499442 ,
        -0.02498822,  0.01327138],
       [-0.05204548, -0.04440248,  0.06644546, ...,  0.2105082 ,
         0.02873536,  0.07778297],
       [-0.15296565, -0.13320123, -0.07977923, ...,  0.0363212 ,
        -0.12891325, -0.07600533]], shape=(95, 278))

In [21]:
## Reunimos datos OG
df_og = pl.DataFrame(og, schema = M_df.columns[1:])
df_og = df_og.with_columns(
    zm = M_df["zm"]
)
df_og = df_og.unpivot(
            cs.numeric(),
            index = "zm"
        ).rename(
            {
                "value" : "og", 
                "variable" : "rama_id"
            }
    )
df_og

zm,rama_id,og
str,str,f64
"""Manzanillo, Col.""","""1125""",-0.136962
"""Metrópoli municipal de Acapulc…","""1125""",-0.014404
"""Metrópoli municipal de Campech…","""1125""",-0.119421
"""Metrópoli municipal de Chetuma…","""1125""",-0.10472
"""Metrópoli municipal de Ciudad …","""1125""",-0.266446
…,…,…
"""Zona metropolitana de Villaher…","""8132""",-0.008408
"""Zona metropolitana de Xalapa""","""8132""",0.068402
"""Zona metropolitana de Zacateca…","""8132""",0.013271


In [22]:
## Reunimos datos Distancia
df_density = pl.DataFrame(density, schema = M_df.columns[1:])
df_density = df_density.with_columns(
    zm = M_df["zm"]
)
df_density = df_density.unpivot(
            cs.numeric(),
            index = "zm"
        ).rename(
            {
                "value" : "density", 
                "variable" : "rama_id"
            }
    )
df_density

zm,rama_id,density
str,str,f64
"""Manzanillo, Col.""","""1125""",0.424509
"""Metrópoli municipal de Acapulc…","""1125""",0.299249
"""Metrópoli municipal de Campech…","""1125""",0.393834
"""Metrópoli municipal de Chetuma…","""1125""",0.338269
"""Metrópoli municipal de Ciudad …","""1125""",0.380237
…,…,…
"""Zona metropolitana de Villaher…","""8132""",0.471269
"""Zona metropolitana de Xalapa""","""8132""",0.402573
"""Zona metropolitana de Zacateca…","""8132""",0.525947


In [23]:
## Reunimos conjunto de datos 
### density y GO
df_portafolios = df_density.join(
    df_og, 
    on = ["zm", "rama_id"]
)

### Agregamos PCI
df_portafolios = df_portafolios.join(
    df_pci.with_columns(pl.col("rama_id").cast(pl.String)), 
    on = "rama_id", 
    how = "left"
)

### Agregamos especialización M
df_portafolios = df_portafolios.join(
    datos_rca_m.select("zm", "rama_id", "M", "transable").with_columns(
        pl.col("rama_id").cast(pl.String)
    ),
    on = ["zm", "rama_id"]
)

### Normalizamos density
df_portafolios = df_portafolios.with_columns(
    density_norm = (pl.col("density") - pl.col("density").mean())/pl.col("density").std()
)
df_portafolios

zm,rama_id,density,og,pci,industria,M,transable,density_norm
str,str,f64,f64,f64,str,i32,i64,f64
"""Zona metropolitana de Aguascal…","""1125""",0.531898,-0.139305,-0.78874,"""Acuicultura""",0,1,1.248192
"""Zona metropolitana de Aguascal…","""1141""",0.520024,-0.135759,-0.672502,"""Pesca""",0,1,1.164649
"""Zona metropolitana de Aguascal…","""1151""",0.587172,-0.090848,-0.177406,"""Servicios relacionados con la …",1,0,1.637104
"""Zona metropolitana de Aguascal…","""1152""",0.579085,-0.038359,0.101975,"""Servicios relacionados con la …",1,0,1.580202
"""Zona metropolitana de Aguascal…","""1153""",0.520623,0.01289,0.313531,"""Servicios relacionados con el …",0,0,1.168862
…,…,…,…,…,…,…,…,…
"""Manzanillo, Col.""","""8123""",0.409597,-0.286682,-1.230191,"""Servicios funerarios y adminis…",1,0,0.387677
"""Manzanillo, Col.""","""8124""",0.347357,-0.307109,-1.183101,"""Estacionamientos y pensiones p…",0,0,-0.050243
"""Manzanillo, Col.""","""8129""",0.340628,0.110708,0.443696,"""Servicios de revelado e impres…",0,0,-0.097594


In [24]:
## Calcula promedio ponderado de density, PCI y GO
### Define ponderadores
product_selection_criteria = {
    "Low-hanging Fruit" : {"og" : 0.25, "pci" : 0.15, "density" : 0.60},
    "Balanced Portfolio" : {"og" : 0.35, "pci" : 0.15, "density" : 0.50},
    "Long Jumps" : {"og" : 0.35, "pci" : 0.20, "density" : 0.45},
}

### Definimos portafolio
portafolio = "Long Jumps"

### Creamos score como una media ponderada
df_portafolios_score = df_portafolios.with_columns(
        df_portafolios.select(
            pl.struct("density_norm", "pci", "og").map_elements(
                lambda s: np.average(
                    a = [s["density_norm"], s["pci"], s["og"]],
                    weights = [
                        product_selection_criteria[portafolio]["density"],
                        product_selection_criteria[portafolio]["pci"], 
                        product_selection_criteria[portafolio]["og"]
                    ]
                ), 
                return_dtype=pl.Float64
            ).alias("score")
        )
    )

df_portafolios_score

zm,rama_id,density,og,pci,industria,M,transable,density_norm,score
str,str,f64,f64,f64,str,i32,i64,f64,f64
"""Zona metropolitana de Aguascal…","""1125""",0.531898,-0.139305,-0.78874,"""Acuicultura""",0,1,1.248192,0.355182
"""Zona metropolitana de Aguascal…","""1141""",0.520024,-0.135759,-0.672502,"""Pesca""",0,1,1.164649,0.342076
"""Zona metropolitana de Aguascal…","""1151""",0.587172,-0.090848,-0.177406,"""Servicios relacionados con la …",1,0,1.637104,0.669419
"""Zona metropolitana de Aguascal…","""1152""",0.579085,-0.038359,0.101975,"""Servicios relacionados con la …",1,0,1.580202,0.71806
"""Zona metropolitana de Aguascal…","""1153""",0.520623,0.01289,0.313531,"""Servicios relacionados con el …",0,0,1.168862,0.593206
…,…,…,…,…,…,…,…,…,…
"""Manzanillo, Col.""","""8123""",0.409597,-0.286682,-1.230191,"""Servicios funerarios y adminis…",1,0,0.387677,-0.171922
"""Manzanillo, Col.""","""8124""",0.347357,-0.307109,-1.183101,"""Estacionamientos y pensiones p…",0,0,-0.050243,-0.366718
"""Manzanillo, Col.""","""8129""",0.340628,0.110708,0.443696,"""Servicios de revelado e impres…",0,0,-0.097594,0.08357


## Para la Zona Metropolitana de Aguascalientes, ¿Cuales son la 10 industrias que no están especializadas con el Score más alto?

In [25]:
### Define zm a analizar
zm_nombre = "Zona metropolitana de Aguascalientes"
#zm_nombre = "Metrópoli municipal de Acapulco"

### Nos quedamos con las industrias para las cuales no se encuentra especializada la región y ordenamos de acuerdo al score
zm_industrias_portafolio = df_portafolios_score.filter(
    (pl.col("zm") == zm_nombre) &
    (pl.col("M") == 0) 
)

zm_industrias_portafolio = zm_industrias_portafolio.select(
    "zm", "rama_id", "industria", "score", "pci"
).sort("score", descending=True)

zm_industrias_portafolio

zm,rama_id,industria,score,pci
str,str,str,f64,f64
"""Zona metropolitana de Aguascal…","""3334""","""Fabricación de equipo de aire …",0.985813,1.398126
"""Zona metropolitana de Aguascal…","""4885""","""Servicios de intermediación pa…",0.962661,1.577363
"""Zona metropolitana de Aguascal…","""5132""","""Edición de software""",0.924394,1.500554
"""Zona metropolitana de Aguascal…","""3342""","""Fabricación de equipo de comun…",0.916308,1.421054
"""Zona metropolitana de Aguascal…","""3364""","""Fabricación de equipo aeroespa…",0.915011,1.27347
…,…,…,…,…
"""Zona metropolitana de Aguascal…","""4611""","""Comercio al por menor de abarr…",-0.07981,-2.098398
"""Zona metropolitana de Aguascal…","""4321""","""Comercio al por mayor de produ…",-0.207972,-2.199526
"""Zona metropolitana de Aguascal…","""3131""","""Preparación e hilado de fibras…",-0.212773,-2.663698


## Grafica relación del PCI con el Score calculado

In [26]:
alt.Chart(zm_industrias_portafolio, title = "ICI vs Score").mark_circle(size=60).encode(
    x=alt.X('pci').title("ICI"),
    y=alt.Y('score').title("Score") ,
    tooltip=['industria', 'pci', 'score']
).interactive()

alt.Chart(...)